# Tests — parameter estimation

Validation digressions split out of the clean pipeline [spatial_analysis_article.ipynb](spatial_analysis_article.ipynb). Here we check that the **vectorized `approx_cdf` pairwise MLE** used in the pipeline is correct and fast:

1. exact scalar `scipy.phi2` baseline on simulated data,
2. classic `scipy.phi2` vs vectorized `approx_cdf` — same $\hat\sigma$, large speed-up,
3. robustness to missing data.

**Every simulated history here is drawn with the Cholesky `simulate_cholesky` (with burn-in)** — the same simulator the pipeline uses (`run_simulation_mle_experiment(..., simulator="cholesky")`, the default). The *scipy/SVD* simulator is examined separately in [tests_simulation_methods.ipynb](tests_simulation_methods.ipynb). The `approx_cdf` kernel itself is derived in [approx_cdf_adaptation/approx_cdf_explained.ipynb](../approx_cdf_adaptation/approx_cdf_explained.ipynb).


## Setup

Rebuild `dict_model_params` exactly as in the pipeline notebook (imports, single-site fit outputs, station subset, exit-probability closures).

In [1]:
import sys, pathlib
_HERE = pathlib.Path.cwd().resolve()
_REPO_ROOT = next(
    (p for p in (_HERE, *_HERE.parents) if (p / "article_code").is_dir()),
    None,
)
if _REPO_ROOT is None:
    raise RuntimeError(f"Could not locate repo root containing 'article_code/' from {_HERE}")
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from article_code.util_files import config
from spatial_bmcd.archive_old_single_latent_field.spatial_model import (
    prepare_station_fit_table,
    build_dict_model_params,
    run_simulation_mle_experiment,
    load_all_station_rr,
    build_joint_df_occurrence_from_raw_data,
    history_from_Rbin_drop_ambiguous_spell_after_nan,
    split_history_by_year,
    mle_sigma_pairwise,
    mle_sigma_pairwise_histories,
    simulate_cholesky,
)
from spatial_bmcd.archive_old_single_latent_field.spatial_plotting import (
    plot_stations_geomap,
    plot_stations_map,
    plot_tail_probability_illustration,
    plot_gaussian_field_with_thresholds,
)

In [2]:
# Load single-site outputs produced by notebook 01.
base_filename = (
    f"ecad_data_south_europe_filtered_after_{config.START_YEAR}"
    f"_wet_day_thresh_{config.WET_DAY_THRESHOLD}.json"
)
with open(config.EXPORTS_JSON_DIR / base_filename) as fh:
    spells = json.load(fh)

stations_metadata = pd.read_csv(config.STATION_METADATA_CSV)
stations_used = stations_metadata[stations_metadata["city"].isin(spells.keys())]

fit_folder = (
    config.RESULTS_FIT_DIR
    / f"fit_south_europe_subset_excess_over_{config.WET_DAY_THRESHOLD}"
)
df_fit_dry = pd.read_csv(fit_folder / "dry_spell_fit_egpd1_excess_over_1result_fit_parameters.csv")
df_fit_wet = pd.read_csv(fit_folder / "wet_spell_fit_mixt_geomresult_fit_parameters.csv")
df_fit_dry["city"] = df_fit_dry["data_source"].map(lambda s: s.split()[0])
df_fit_dry["season"] = df_fit_dry["data_source"].map(lambda s: s.split()[-1])

print(f"{len(spells)} stations in spells JSON; {len(stations_used)} in metadata; "
      f"{df_fit_dry['city'].nunique()} cities in dry fit, "
      f"{df_fit_wet['city'].nunique()} in wet fit")

209 stations in spells JSON; 209 in metadata; 209 cities in dry fit, 209 in wet fit


In [3]:
COUNTRIES = ["PT", "ES"]
SEASON = "spring"

df_fit_dry_sub, df_fit_wet_sub = prepare_station_fit_table(
    df_fit_dry, df_fit_wet, stations_used,
    countries=COUNTRIES, season=SEASON,
)
print(f"{len(df_fit_dry_sub)} stations kept for season={SEASON}, countries={COUNTRIES}")

plot_stations_geomap(df_fit_dry_sub).show()
# plot_stations_map(df_fit_dry_sub, color_col="xi");

35 stations kept for season=spring, countries=['PT', 'ES']


In [4]:
dict_model_params = build_dict_model_params(
    df_fit_dry_sub, df_fit_wet_sub, spells, season=SEASON,
)
print(f"Built model parameters for {len(dict_model_params)} stations")

# Sanity check: q^(0)(1), q^(0)(5), q^(1)(1) for the first station
city0 = next(iter(dict_model_params))
p = dict_model_params[city0]
print(f"{city0}: q_dry(1)={p['q_d_dry_function'](1):.3f}, "
      f"q_dry(5)={p['q_d_dry_function'](5):.3f}, "
      f"q_dry(20)={p['q_d_dry_function'](20):.3f}, --- "
      f"q_wet(1)={p['q_d_wet_function'](1):.3f}",
      f"q_wet(5)={p['q_d_wet_function'](5):.3f}",
      f"q_wet(20)={p['q_d_wet_function'](20):.3f}")

  0%|          | 0/35 [00:00<?, ?it/s]

 60%|██████    | 21/35 [00:00<00:00, 205.23it/s]

100%|██████████| 35/35 [00:00<00:00, 176.14it/s]

Built model parameters for 35 stations
ALACANT/ALICANTE: q_dry(1)=0.142, q_dry(5)=0.101, q_dry(20)=0.074, --- q_wet(1)=0.538 q_wet(5)=0.521 q_wet(20)=0.441


## Maximum-likelihood estimation on simulated data

Sanity-check the pairwise composite likelihood by simulating spatial histories with a known $\sigma$ and recovering it via `run_simulation_mle_experiment`. Each row of the returned DataFrame is one independent simulation.

In [5]:
SIGMA_TRUE = 0.2
# "Old simple" estimator: exact scipy phi2 in a scalar pair loop (the default path).
# Kept deliberately small -- it is correct but slow, which is exactly what we fix below.
EXPERIMENTS = [
    dict(nb_stations=5, nb_steps=150, nb_estimations=3),
    dict(nb_stations=8, nb_steps=100, nb_estimations=2),
]

results = []
for cfg in EXPERIMENTS:
    df = run_simulation_mle_experiment(
        dict_model_params, sigma_true=SIGMA_TRUE, **cfg,
    )
    df["cfg"] = f"J={cfg['nb_stations']}, T={cfg['nb_steps']}"
    results.append(df)
    print(f"\n{df['cfg'].iloc[0]} -- sigma_true={SIGMA_TRUE}")
    print(df[["i", "sigma_hat", "ll_hat"]].round(4).to_string(index=False))

results_df = pd.concat(results, ignore_index=True)


J=5, T=150 -- sigma_true=0.2
 i  sigma_hat     ll_hat
 0     0.1923 -1537.7439
 1     0.1855 -1539.8615
 2     0.2295 -1576.6495



J=8, T=100 -- sigma_true=0.2
 i  sigma_hat     ll_hat
 0     0.2056 -2663.3860
 1     0.2579 -2415.8693


## Fast likelihood with the vectorized `approx_cdf`

The estimator above is correct but slow: its bottleneck is the bivariate-normal CDF `phi2`, which rebuilds a fresh `scipy.stats.multivariate_normal` object for every station pair inside the composite-likelihood loop. We replace it with `phi2_vec` — a closed-form, dependency-free port of the Tsay & Ke (2021) approximation that vectorizes over all pairs of a time step at once. The derivation and accuracy checks live in [approx_cdf_adaptation/approx_cdf_explained.ipynb](../approx_cdf_adaptation/approx_cdf_explained.ipynb).

All likelihood/MLE functions now take `vectorized=True` to switch to this kernel. On the **same** simulated history below, it recovers the same $\hat\sigma$ (the ~$10^{-3}$ CDF error moves $\hat\sigma$ by ~$10^{-5}$) but runs much faster.

In [6]:
import time

# Head-to-head on ONE simulated history. We use a 12-station subset so the (slow)
# scalar baseline stays tractable here; the vectorized path handles the full 35
# stations effortlessly and is what every cell below uses.
cmp_params = {c: dict_model_params[c] for c in sorted(dict_model_params)[:12]}
sim_hist = simulate_cholesky(
    sigma=SIGMA_TRUE, params_by_station=cmp_params,
    n_steps=120, n_burn=150, seed=1,
)

t0 = time.perf_counter()
r_slow = mle_sigma_pairwise(sim_hist, cmp_params, bounds=(1e-3, 2.0))
t_slow = time.perf_counter() - t0

t0 = time.perf_counter()
r_fast = mle_sigma_pairwise(sim_hist, cmp_params, bounds=(1e-3, 2.0), vectorized=True)
t_fast = time.perf_counter() - t0

d_sigma = abs(r_fast["sigma_hat"] - r_slow["sigma_hat"])
print(f"sigma_true = {SIGMA_TRUE}   (J={len(cmp_params)} stations, T={sim_hist['S'].shape[0]})")
print(f"  scipy phi2 (scalar) : sigma_hat={r_slow['sigma_hat']:.5f}  ll={r_slow['ll_hat']:.2f}  ({t_slow:.2f}s)")
print(f"  approx_cdf (vector) : sigma_hat={r_fast['sigma_hat']:.5f}  ll={r_fast['ll_hat']:.2f}  ({t_fast:.2f}s)")
print(f"  |d sigma_hat| = {d_sigma:.2e}  (relative {d_sigma / r_slow['sigma_hat']:.2%})")
print(f"  speedup       = {t_slow / t_fast:.0f}x")

sigma_true = 0.2   (J=12 stations, T=120)
  scipy phi2 (scalar) : sigma_hat=0.19279  ll=-6430.44  (39.07s)
  approx_cdf (vector) : sigma_hat=0.19272  ll=-6430.40  (0.97s)
  |d sigma_hat| = 6.87e-05  (relative 0.04%)
  speedup       = 40x


### Same check through `run_simulation_mle_experiment`

We repeat the classic-vs-`approx_cdf` comparison through `run_simulation_mle_experiment`. Both calls use `simulator="cholesky"` (the default), so with the same `seed_base` they generate the **identical** Cholesky-drawn histories and the only thing that differs is the likelihood kernel.

In [7]:
# Same classic-vs-approx comparison, through run_simulation_mle_experiment.
# Both calls use simulator="cholesky" (the default): histories are drawn by
# simulate_cholesky (Sigma = L L^T factorisation, with burn-in) -- the same
# Cholesky path as the clean pipeline notebook. With the same seed_base both calls
# simulate the SAME histories, so any sigma_hat difference is purely the likelihood
# kernel (scipy phi2 vs vectorized approx_cdf).
cfg = dict(nb_stations=8, nb_steps=100, nb_estimations=3)

t0 = time.perf_counter()
df_classic = run_simulation_mle_experiment(
    dict_model_params, sigma_true=SIGMA_TRUE, seed_base=0, simulator="cholesky", **cfg)
t_classic = time.perf_counter() - t0

t0 = time.perf_counter()
df_approx = run_simulation_mle_experiment(
    dict_model_params, sigma_true=SIGMA_TRUE, seed_base=0, simulator="cholesky",
    vectorized=True, **cfg)
t_approx = time.perf_counter() - t0

cmp = df_classic[["i", "sigma_hat"]].rename(columns={"sigma_hat": "sigma_hat_classic"})
cmp["sigma_hat_approx"] = df_approx["sigma_hat"].values
cmp["abs_diff"] = (cmp["sigma_hat_classic"] - cmp["sigma_hat_approx"]).abs()
print(f"sigma_true={SIGMA_TRUE}, J={cfg['nb_stations']}, T={cfg['nb_steps']} "
      f"-- identical Cholesky histories (seed_base=0), {cfg['nb_estimations']} simulations")
print(cmp.round(5).to_string(index=False))
print(f"\nclassic (scipy phi2)    total: {t_classic:.2f}s")
print(f"approx_cdf (vectorized) total: {t_approx:.2f}s   speedup {t_classic / t_approx:.1f}x")


sigma_true=0.2, J=8, T=100 -- identical Cholesky histories (seed_base=0), 3 simulations
 i  sigma_hat_classic  sigma_hat_approx  abs_diff
 0            0.20561           0.20584   0.00023
 1            0.25794           0.25804   0.00009
 2            0.15219           0.15203   0.00016

classic (scipy phi2)    total: 47.47s
approx_cdf (vectorized) total: 2.77s   speedup 17.1x


## Robustness to missing data

Same experiment, but with 10% of stations randomly masked at every time step. The pairwise likelihood is NaN-aware, so the estimator should still recover $\sigma$.

In [8]:
df_nan = run_simulation_mle_experiment(
    dict_model_params, sigma_true=SIGMA_TRUE,
    nb_stations=5, nb_steps=100, nb_estimations=5,
    inject_nan_frac=0.10, vectorized=True,
)
print(f"sigma_true={SIGMA_TRUE}, frac_nan=0.10")
print(df_nan[["i", "sigma_hat", "ll_hat", "mean_nb_obs_stations"]].round(4).to_string(index=False))

sigma_true=0.2, frac_nan=0.10
 i  sigma_hat    ll_hat  mean_nb_obs_stations
 0     0.1915 -585.1006                   4.0
 1     0.1216 -653.1772                   4.0
 2     0.2083 -740.3545                   4.0
 3     0.1704 -505.1436                   4.0
 4     0.1984 -485.4756                   4.0


### Scratch

Leftover quick-look cell from the original notebook (kept so nothing is lost; harmless to skip).

In [9]:
# Scratch quick-look kept from the original notebook. It inspects `Rbin`, which is
# built in the pipeline real-data section (spatial_analysis_article.ipynb), not here --
# guarded so this tests notebook still runs top to bottom.
if "Rbin" in globals():
    Rbin.columns
